# Camada Analítica (Business Answers)

Nesta camada (Gold Layer), utilizamos a tabela limpa e padronizada proveniente da **Camada Silver** (`vw_silver_nyc_taxi_consumption`) para responder às perguntas de negócio do case. 

Para garantir a acurácia dos dados financeiros e de operação, foram aplicadas as seguintes premissas analíticas:
* Valores totais (`total_amount`) negativos ou zerados foram removidos das agregações financeiras para não distorcer a média com estornos de cartão.
* Contagens de passageiros inválidas foram removidas da média horária.
* Uma coluna de volume (`total_trips`) foi adicionada lado a lado com a média para dar visibilidade da significância da amostra.

## Pergunta 1
**Qual a média do `total_amount` por mês (apenas para a frota Yellow)?**

In [0]:
from pyspark.sql.functions import col, round, avg, count

# 1. Filtro: Frota Yellow, com tarifa válida (maior que zero)
df_q1 = spark.table("vw_silver_nyc_taxi_consumption").filter(
    (col("taxi_type") == "yellow") & 
    (col("total_amount").isNotNull()) & 
    (col("total_amount") > 0)
)

# 2. Agrupamento por mês e cálculos matemáticos
ans_q1 = df_q1.groupBy("pickup_month") \
              .agg(
                  round(avg("total_amount"), 2).alias("avg_total_amount_usd"),
                  count("*").alias("total_trips")
              ) \
              .orderBy("pickup_month")

# Exibe o resultado da Pergunta 1
display(ans_q1)

pickup_month,avg_total_amount_usd,total_trips
1,27.45,3039875
2,27.34,2887317
3,28.27,3371881
4,28.76,3256849
5,29.46,3480251


## Pergunta 2
**Qual a média do `passenger_count` por hora do dia para as viagens que aconteceram em Maio (Para TODOS os Táxis)?**

In [0]:
from pyspark.sql.functions import hour

# 1. Filtro: Apenas Mês 5 (Maio) e contagem de passageiros válida
# Obs: Não filtramos por taxi_type pois a requisição exige "All Taxis"
df_q2 = spark.table("vw_silver_nyc_taxi_consumption").filter(
    (col("pickup_month") == 5) & 
    (col("passenger_count").isNotNull()) & 
    (col("passenger_count") > 0)
)

# 2. Extração da "Hora do Dia" a partir da data de início da viagem
df_q2 = df_q2.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

# 3. Agrupamento por hora (0 a 23h) e cálculos
ans_q2 = df_q2.groupBy("pickup_hour") \
              .agg(
                  round(avg("passenger_count"), 2).alias("avg_passenger_count"),
                  count("*").alias("total_trips_in_hour")
              ) \
              .orderBy("pickup_hour")

# Exibe o resultado da Pergunta 2
display(ans_q2)


pickup_hour,avg_passenger_count,total_trips_in_hour
0,1.43,90817
1,1.43,59122
2,1.45,38166
3,1.45,25042
4,1.4,16475
5,1.28,18850
6,1.26,46919
7,1.28,94738
8,1.29,129158
9,1.31,145051
